In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
from sklearn.model_selection import GridSearchCV
import re 
import joblib
import string

In [2]:
fake=pd.read_csv('./dataset/Fake.csv')
true=pd.read_csv('./dataset/True.csv')

In [3]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
fake['class']=0
true['class']=1

In [6]:
data=pd.concat([fake,true],axis=0)

In [7]:
data.sample(10)

,title,text,subject,date,class
14981,Not Kidding! Obama’s Dept. Of Education Orders...,This news is shocking and I m glad my kids don...,politics,"Nov 3, 2015",0
16456,"Qatar, Russia sign agreements on air defense, ...",DOHA (Reuters) - Qatar signed a military techn...,worldnews,"October 26, 2017",1
5644,RNC BANS Watchdog That Has Attended The GOP C...,The Republican National Committee seems to be ...,News,"July 2, 2016",0
21509,“BLOOD ON THEIR HANDS” FOR VOTING RIGHTS: The ...,It s time to stop hitting the snooze button Am...,left-news,"Aug 9, 2015",0
966,Trump likely to pick Fed's Powell to lead cent...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"October 30, 2017",1
4019,Report: Hillary Wants Joe Biden For HUGE Role...,A new report indicates that Vice President Joe...,News,"October 27, 2016",0
9838,Senators close to proposal on Zika funds: Repu...,WASHINGTON (Reuters) - A bipartisan group of s...,politicsNews,"April 21, 2016",1
3922,Tim Kaine Reveals Who’s Trying To Steal The E...,"Senator Tim Kaine, the Democratic Party s vice...",News,"November 5, 2016",0
17675,Kenya bans city-center protests as vote tensio...,NAIROBI (Reuters) - Kenyan authorities banned ...,worldnews,"October 12, 2017",1
16370,WIKILEAKS Posts NEW Document…Shows Hillary Rep...,Hillary Clinton is the last person you want pl...,Government News,"Oct 3, 2016",0


In [8]:
data=data.drop(['title','subject','date'],axis=1)

In [9]:
data.head()

,text,class
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0


In [10]:
data.reset_index(inplace=True)

In [11]:
data.drop(['index'],axis=1,inplace=True)

In [12]:
data.sample(5)

,text,class
19962,"Former Senate President of Haiti, Bernard Sans...",0
28676,(Reuters) - President Donald Trump told Congre...,1
39970,DUBAI (Reuters) - Seven suspected al Qaeda mil...,1
15766,"Quicken Loans owner, Dan Gilbert is one of the...",0
44654,"BOSSASO, Somalia (Reuters) - An al Shabaab bom...",1


In [13]:
def clean(text):
    text=text.lower()
    text=re.sub(r"\[.*?\]","",text)
    text=re.sub(r"\\W"," ",text)
    text=re.sub(r"https?://\S+|www\.\S+","",text)
    text=re.sub(r"<.*?>+","",text)
    text=re.sub(r"[%s]" % re.escape(string.punctuation),"",text)
    text=re.sub(r"\n","",text)
    text=re.sub(r"\w*\d\w*","",text)
    return text


In [14]:
data['text']=data['text'].apply(clean)

In [15]:
x=data['text']
y=data['class']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [ ]:
#vectorizer=TfidfVectorizer()


In [16]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [17]:
xv_train=tfidf.fit_transform(x_train)
xv_test=tfidf.transform(x_test)

In [ ]:
#xv_train=vectorizer.fit_transform(x_train)
#xv_test=vectorizer.transform(x_test)

GridSearchCV for LogisticRegression

In [18]:
# GridSearchCV
lr_params={
    "C":[0.1,1,2]
}

lr_gridcv=GridSearchCV(LogisticRegression(max_iter=3000),lr_params,cv=5,scoring="f1",n_jobs=-1)


In [19]:
lr_gridcv.fit(xv_train,y_train)
lr_model=lr_gridcv.best_estimator_

print("Best LR params :",lr_gridcv.best_params_)

Best LR params : {'C': 2}


GridSearchCV for NaiveBayse

In [46]:
# =====================================
# Naive Bayes
# =====================================

print("\nTraining Naive Bayes...\n")

nb_params = {
    "alpha": [0.01, 0.1, 0.5, 1]
}

nb_grid = GridSearchCV(
    MultinomialNB(),
    nb_params,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

nb_grid.fit(xv_train,y_train)

nb_model = nb_grid.best_estimator_

print("Best NB Params:",nb_grid.best_params_)



Training Naive Bayes...

Best NB Params: {'alpha': 0.01}


GridSearchCV for RandomForest

In [47]:

# =====================================
# Random Forest
# =====================================

print("\nTraining Random Forest...\n")

rf_params = {
    "n_estimators": [200, 300],
    "max_depth": [20, 30, None],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params,cv=3, scoring="f1", n_jobs=-1)
    

rf_grid.fit( xv_train, y_train)

rf_model = rf_grid.best_estimator_

print("Best RF Params:")
print(rf_grid.best_params_)



Training Random Forest...

Best RF Params:
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}


In [20]:
# =====================================
# Evaluation Function
# =====================================
def evaluate_model( model,X_test,y_test,model_name):
   
    predictions = model.predict(X_test)
            
    accuracy = accuracy_score(y_test,predictions)

    precision = precision_score( y_test,predictions)   

    recall = recall_score(y_test,predictions )

    f1 = f1_score(y_test, predictions) 

    print("\n")
    print("=" * 50)
    print(model_name)
    print("=" * 50)            
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(classification_report(y_test,predictions))

    return accuracy           
                    

In [21]:
lr_acc=evaluate_model(lr_model,xv_test,y_test,"Logistic Regression")




Logistic Regression
Accuracy : 0.9911
Precision: 0.9897
Recall   : 0.9916
F1 Score : 0.9906
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.99      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [ ]:
nb_acc=evaluate_model(nb_model,xv_test,y_test,"Naive Bayes")
rf_acc=evaluate_model(rf_model,xv_test,y_test,"Random Forest")

Saving Models

In [22]:
print("\nSaving Models............\n")

joblib.dump(tfidf,"./models/vectorizer.jb")
joblib.dump(lr_model,"./models/logistic_regression.jb")




Saving Models............



['./models/logistic_regression.jb']

In [ ]:
joblib.dump(nb_model,"./models/naive_bayes.jb")
joblib.dump(rf_model,"./models/random_forest.jb")

print("Models Saved Successfully")

In [20]:
lr=LogisticRegression()
lr.fit(xv_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [21]:
prediction=lr.predict(xv_test)
lr.score(xv_test,y_test)

0.985924276169265

In [22]:
print(classification_report(y_test,prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [23]:
#joblib.dump(vectorizer,"./models/vectorizer.jb")
#joblib.dump(lr,"./models/lr_model.jb")